# Experimento 03 — Remoção do `cibil_score`

**Dataset:** [Loan Approval Prediction Dataset](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset) · **Algoritmo:** `DecisionTreeClassifier` · **Tarefa:** classificação binária (`Approved` x `Rejected`)

Árvore de decisão treinada com dados limpos e **sem** o `cibil_score` (score de crédito), para avaliar o que as demais variáveis conseguem prever sozinhas.

| Preparação dos dados | Aplicada |
|---|:-:|
| Tratar valores inválidos (remover linhas) | ✅ |
| Remover variáveis fracas | — |
| Remover `cibil_score` | ✅ |
| Undersampling no treino | — |

## Importação das bibliotecas

In [ ]:
# !pip install kagglehub scikit-learn imbalanced-learn pandas matplotlib seaborn  # descomente se necessário

import glob
import os

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    learning_curve,
    train_test_split,
    validation_curve,
)
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

RANDOM_STATE = 42
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
CLASSES = ["Rejected", "Approved"]
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Configuração do cenário

Altere estes valores para testar outras variações.

In [ ]:
NOME_CENARIO = "03 · Sem cibil_score"
REMOVER_COLUNAS = ["cibil_score"]
UNDERSAMPLING = False
ARQUIVO_METRICAS = "metricas_03_sem_cibil_score.csv"

## Carregamento e codificação dos dados

In [ ]:
path = kagglehub.dataset_download("architsharma01/loan-approval-prediction-dataset")
csv_path = glob.glob(os.path.join(path, "*.csv"))[0]
df = pd.read_csv(csv_path)

# Os nomes das colunas e os valores de texto vêm com espaços no início
df.columns = df.columns.str.strip()
for col in df.select_dtypes(exclude="number").columns:
    df[col] = df[col].str.strip()

data = df.drop(columns=["loan_id"]).copy()
data["education"] = data["education"].map({"Graduate": 1, "Not Graduate": 0})
data["self_employed"] = data["self_employed"].map({"Yes": 1, "No": 0})
data["loan_status"] = data["loan_status"].map({"Approved": 1, "Rejected": 0})
assert data.isnull().sum().sum() == 0, "Há valores não mapeados!"

print("Formato:", data.shape)
data["loan_status"].value_counts().rename(index={1: "Approved", 0: "Rejected"})

## Tratamento de valores inválidos

Verificamos valores que não fazem sentido no domínio do problema:

- **patrimônio negativo** em qualquer uma das 4 colunas de ativos;
- `cibil_score` fora da faixa válida do score (300 a 900);
- prazo, renda ou valor do empréstimo menores ou iguais a zero;
- linhas duplicadas (todas as variáveis iguais).

As linhas com qualquer um desses problemas são **removidas**.

In [ ]:
COLS_ATIVOS = ["residential_assets_value", "commercial_assets_value",
               "luxury_assets_value", "bank_asset_value"]

checagens = {
    "patrimônio negativo": (data[COLS_ATIVOS] < 0).any(axis=1),
    "cibil_score fora de 300–900": ~data["cibil_score"].between(300, 900),
    "loan_term <= 0": data["loan_term"] <= 0,
    "income_annum <= 0": data["income_annum"] <= 0,
    "loan_amount <= 0": data["loan_amount"] <= 0,
    "linha duplicada": data.duplicated(),
}
invalido = pd.concat(checagens, axis=1).any(axis=1)

print(pd.Series({k: int(v.sum()) for k, v in checagens.items()}, name="linhas").to_string())
print(f"\nTotal de linhas inválidas: {invalido.sum()} ({invalido.mean():.2%} do dataset)")
print("\nValores negativos por coluna de ativos:")
print((data[COLS_ATIVOS] < 0).sum().to_string())

In [ ]:
# Proporção das classes no dataset completo e nas linhas removidas
pd.DataFrame({
    "todas as linhas": data["loan_status"].value_counts(normalize=True),
    "linhas removidas": data.loc[invalido, "loan_status"].value_counts(normalize=True),
}).rename(index={1: "Approved", 0: "Rejected"}).round(3)

## Divisão treino/teste

A divisão 80/20 (estratificada, `random_state=42`) é feita sobre o dataset **completo** e só depois as linhas inválidas são retiradas do treino e do teste. Assim, todos os notebooks de experimento usam **exatamente as mesmas linhas de teste**, o que permite comparar os resultados entre eles depois.

In [ ]:
idx_treino, idx_teste = train_test_split(
    data.index, test_size=0.2, stratify=data["loan_status"], random_state=RANDOM_STATE
)
validos = data.index[~invalido]
idx_treino = idx_treino.intersection(validos)
idx_teste = idx_teste.intersection(validos)

COLUNAS = [c for c in data.columns if c not in REMOVER_COLUNAS + ["loan_status"]]
X_train, y_train = data.loc[idx_treino, COLUNAS], data.loc[idx_treino, "loan_status"]
X_test, y_test = data.loc[idx_teste, COLUNAS], data.loc[idx_teste, "loan_status"]

print("Variáveis usadas:", len(COLUNAS))
print(COLUNAS)
print("\nTreino:", X_train.shape, "| Teste:", X_test.shape)
print("Proporção de aprovados — treino: %.3f | teste: %.3f" % (y_train.mean(), y_test.mean()))

## Preparação específica do cenário

### Variável removida: `cibil_score`

Distribuição do score de crédito para cada classe. Quanto menos as duas distribuições se sobrepõem, mais informação a variável carrega, e mais o modelo perde ao ficar sem ela.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
dados_validos = data.loc[validos]
for valor, nome, cor in [(0, "Rejected", CORES[1]), (1, "Approved", CORES[0])]:
    ax.hist(dados_validos.loc[dados_validos["loan_status"] == valor, "cibil_score"],
            bins=30, alpha=0.6, color=cor, label=nome)
ax.set_xlabel("cibil_score")
ax.set_ylabel("Número de pedidos")
ax.set_title("Distribuição do cibil_score por classe")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Treino com ajuste de hiperparâmetros (GridSearchCV)

Validação cruzada estratificada de 5 folds, otimizando o **F1-score**.

In [ ]:
PARAM_GRID = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 4, 5, 6, 8, 10, None],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "class_weight": [None, "balanced"],
}
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

etapas = [("arvore", DecisionTreeClassifier(random_state=RANDOM_STATE))]
grade = {f"arvore__{k}": v for k, v in PARAM_GRID.items()}
if UNDERSAMPLING:
    etapas.insert(0, ("under", RandomUnderSampler(random_state=RANDOM_STATE)))
    del grade["arvore__class_weight"]

grid = GridSearchCV(Pipeline(etapas), grade, scoring="f1", cv=CV, n_jobs=-1)
grid.fit(X_train, y_train)

modelo = grid.best_estimator_
arvore = modelo.named_steps["arvore"]

print("Melhores hiperparâmetros:")
for k, v in grid.best_params_.items():
    print(f"  {k.removeprefix('arvore__')}: {v}")
print("\nF1 na validação cruzada: %.4f ± %.4f"
      % (grid.best_score_, grid.cv_results_["std_test_score"][grid.best_index_]))
print("Profundidade: %d | Folhas: %d" % (arvore.get_depth(), arvore.get_n_leaves()))

In [ ]:
# As 10 melhores combinações de hiperparâmetros
resultados_cv = pd.DataFrame(grid.cv_results_)
resultados_cv["params"] = resultados_cv["params"].apply(
    lambda p: {k.removeprefix("arvore__"): v for k, v in p.items()})
resultados_cv[["params", "mean_test_score", "std_test_score", "rank_test_score"]] \
    .sort_values("rank_test_score").head(10)

## Avaliação no conjunto de teste

### Métricas

- `recall_aprovado`: fração dos pedidos aprovados que o modelo aprovou.
- `recall_rejeitado`: fração dos pedidos rejeitados que o modelo rejeitou.
- `acurácia_balanceada`: média dos dois recalls, menos afetada pelo desbalanceamento das classes.
- `precisão_média`: área sob a curva precisão-recall.

In [ ]:
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

metricas = pd.Series({
    "linhas_treino": len(y_train),
    "linhas_treino_efetivas": 2 * y_train.value_counts().min() if UNDERSAMPLING else len(y_train),
    "nº_variáveis": len(COLUNAS),
    "acurácia_treino": accuracy_score(y_train, modelo.predict(X_train)),
    "acurácia_teste": accuracy_score(y_test, y_pred),
    "acurácia_balanceada": balanced_accuracy_score(y_test, y_pred),
    "precisão": precision_score(y_test, y_pred),
    "recall_aprovado": recall_score(y_test, y_pred),
    "recall_rejeitado": recall_score(y_test, y_pred, pos_label=0),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "precisão_média": average_precision_score(y_test, y_proba),
    "f1_cv_média": grid.best_score_,
    "f1_cv_desvio": grid.cv_results_["std_test_score"][grid.best_index_],
    "profundidade": arvore.get_depth(),
    "nº_folhas": arvore.get_n_leaves(),
}, name=NOME_CENARIO)

metricas.round(4).to_frame()

In [ ]:
# Salva as métricas deste cenário em CSV, para consulta posterior
metricas.to_frame().T.to_csv(ARQUIVO_METRICAS)
print("Métricas salvas em", ARQUIVO_METRICAS)

### Classification report

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Rejected (0)", "Approved (1)"], digits=4))

### Matriz de confusão

À esquerda, a contagem de pedidos; à direita, a proporção por classe real (cada linha soma 1). O erro na célula *Rejected → Approved* (aprovar um pedido que deveria ser rejeitado) costuma ser o mais caro para o negócio.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=CLASSES, cmap="Blues", colorbar=False, ax=axes[0])
axes[0].set_title("Contagem")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=CLASSES, cmap="Blues", colorbar=False,
    normalize="true", values_format=".2%", ax=axes[1])
axes[1].set_title("Proporção por classe real")
for ax in axes:
    ax.grid(False)
    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe real")
fig.suptitle(f"Matriz de confusão — {NOME_CENARIO}")
plt.tight_layout()
plt.show()

### Curva ROC e curva precisão-recall

A curva ROC mostra a troca entre a taxa de verdadeiros positivos e a de falsos positivos para todos os limiares de decisão; a área sob ela é o ROC-AUC. A curva precisão-recall é mais informativa quando as classes são desbalanceadas. Linhas tracejadas: desempenho de um classificador aleatório.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
precisao_curva, recall_curva, _ = precision_recall_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, linewidth=2, color=CORES[0], label=f"Árvore de decisão (AUC = {auc:.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório (AUC = 0.5)")
axes[0].set_xlabel("Taxa de falsos positivos")
axes[0].set_ylabel("Taxa de verdadeiros positivos")
axes[0].set_title("Curva ROC")
axes[0].legend(loc="lower right", frameon=False)

axes[1].step(recall_curva, precisao_curva, where="post", linewidth=2, color=CORES[0],
             label=f"Árvore de decisão (AP = {ap:.3f})")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", label=f"Aleatório (AP = {y_test.mean():.2f})")
axes[1].set_xlabel("Recall (Approved)")
axes[1].set_ylabel("Precisão (Approved)")
axes[1].set_title("Curva precisão-recall")
axes[1].legend(loc="lower left", frameon=False)

for ax in axes:
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

### Distribuição das probabilidades previstas

Probabilidade de aprovação atribuída pelo modelo a cada pedido de teste, separada pela classe real. Um bom modelo concentra os `Rejected` perto de 0 e os `Approved` perto de 1. Como a árvore atribui a mesma probabilidade a todos os pedidos de uma folha, os valores aparecem agrupados.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
faixas = np.linspace(0, 1, 21)
ax.hist(y_proba[y_test == 0], bins=faixas, alpha=0.6, color=CORES[1], label="Rejected (real)")
ax.hist(y_proba[y_test == 1], bins=faixas, alpha=0.6, color=CORES[0], label="Approved (real)")
ax.axvline(0.5, linestyle="--", color="gray", label="limiar padrão (0,5)")
ax.set_xlabel("Probabilidade prevista de aprovação")
ax.set_ylabel("Número de pedidos")
ax.set_title("Probabilidades previstas por classe real")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Efeito do limiar de decisão

Por padrão, um pedido é aprovado quando a probabilidade prevista é ≥ 0,5. Subir o limiar torna a aprovação mais rigorosa: aumenta o `recall_rejeitado` e reduz o `recall_aprovado`. O gráfico mostra essa troca sem precisar treinar outro modelo.

In [ ]:
limiares = np.round(np.arange(0.05, 1.0, 0.05), 2)
curva_limiar = pd.DataFrame([{
    "limiar": t,
    "precisão": precision_score(y_test, (y_proba >= t).astype(int), zero_division=0),
    "recall_aprovado": recall_score(y_test, (y_proba >= t).astype(int)),
    "recall_rejeitado": recall_score(y_test, (y_proba >= t).astype(int), pos_label=0),
    "f1": f1_score(y_test, (y_proba >= t).astype(int), zero_division=0),
} for t in limiares]).set_index("limiar")

fig, ax = plt.subplots(figsize=(10, 5))
for cor, col in zip(CORES, curva_limiar.columns):
    ax.plot(curva_limiar.index, curva_limiar[col], marker="o", markersize=5, linewidth=2, color=cor, label=col)
ax.axvline(0.5, linestyle="--", color="gray")
ax.set_xlabel("Limiar de decisão")
ax.set_ylabel("Valor da métrica")
ax.set_title("Métricas em função do limiar de decisão")
ax.legend(frameon=False, loc="lower left")
plt.tight_layout()
plt.show()

curva_limiar.round(4)

## Estabilidade na validação cruzada

O modelo escolhido é reavaliado em cada um dos 5 folds do treino. Pouca dispersão entre os folds indica um modelo estável, que não depende muito de quais linhas caíram no treino.

In [ ]:
validacao = cross_validate(
    modelo, X_train, y_train, cv=CV, n_jobs=-1,
    scoring={"acurácia": "accuracy", "precisão": "precision", "recall": "recall",
             "f1": "f1", "roc_auc": "roc_auc"},
)
folds = pd.DataFrame({k.removeprefix("test_"): v for k, v in validacao.items() if k.startswith("test_")})
folds.index = [f"fold {i + 1}" for i in range(len(folds))]

fig, ax = plt.subplots(figsize=(9, 4))
for i, col in enumerate(folds.columns):
    ax.scatter(folds[col], [i] * len(folds), s=60, color=CORES[0], alpha=0.7, zorder=3)
    ax.scatter(folds[col].mean(), i, s=160, marker="|", color="black", zorder=4)
ax.set_yticks(range(len(folds.columns)), folds.columns)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("Valor no fold (traço preto = média)")
ax.set_title("Métricas em cada fold da validação cruzada")
plt.tight_layout()
plt.show()

pd.concat([folds, folds.agg(["mean", "std"]).rename(index={"mean": "média", "std": "desvio"})]).round(4)

## Curva de aprendizado

F1 no treino e na validação cruzada à medida que o modelo recebe mais linhas de treino. Se a curva de validação ainda sobe no final, mais dados ajudariam; uma distância grande entre as duas curvas indica *overfitting*.

In [ ]:
tamanhos, f1_treino, f1_validacao = learning_curve(
    modelo, X_train, y_train, cv=CV, scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(9, 5))
for valores, nome, cor in [(f1_treino, "Treino", CORES[0]), (f1_validacao, "Validação", CORES[1])]:
    media, desvio = valores.mean(axis=1), valores.std(axis=1)
    ax.plot(tamanhos, media, marker="o", linewidth=2, color=cor, label=nome)
    ax.fill_between(tamanhos, media - desvio, media + desvio, color=cor, alpha=0.15)
rotulo_x = "Linhas de treino" + (" (antes do undersampling)" if UNDERSAMPLING else "")
ax.set_xlabel(rotulo_x)
ax.set_ylabel("F1")
ax.set_title("Curva de aprendizado (média ± desvio nos folds)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Curva de complexidade (profundidade da árvore)

Mantendo os demais hiperparâmetros escolhidos, variamos apenas `max_depth`. A linha tracejada marca a profundidade do modelo final.

In [ ]:
profundidades = list(range(1, 16))
f1_treino, f1_validacao = validation_curve(
    modelo, X_train, y_train, param_name="arvore__max_depth", param_range=profundidades,
    cv=CV, scoring="f1", n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(9, 5))
for valores, nome, cor in [(f1_treino, "Treino", CORES[0]), (f1_validacao, "Validação", CORES[1])]:
    media, desvio = valores.mean(axis=1), valores.std(axis=1)
    ax.plot(profundidades, media, marker="o", linewidth=2, color=cor, label=nome)
    ax.fill_between(profundidades, media - desvio, media + desvio, color=cor, alpha=0.15)
ax.axvline(arvore.get_depth(), linestyle="--", color="gray", label=f"modelo final (profundidade {arvore.get_depth()})")
ax.set_xlabel("max_depth")
ax.set_ylabel("F1")
ax.set_title("F1 em função da profundidade da árvore")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Interpretação do modelo

### Importância das variáveis

In [ ]:
importancias = pd.Series(arvore.feature_importances_, index=COLUNAS).sort_values()

fig, ax = plt.subplots(figsize=(9, 0.45 * len(COLUNAS) + 1.5))
ax.barh(importancias.index, importancias.values, color=CORES[0])
for i, v in enumerate(importancias.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0, min(1.0, importancias.max() + 0.1))
ax.set_xlabel("Importância (redução de impureza)")
ax.set_title("Importância das variáveis")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### Visualização da árvore

In [ ]:
# Exibe no máximo os 4 primeiros níveis para manter a figura legível
plt.figure(figsize=(26, 11))
plot_tree(
    arvore,
    feature_names=COLUNAS,
    class_names=CLASSES,
    filled=True,
    rounded=True,
    max_depth=4,
    fontsize=9,
)
plt.title(f"Árvore de decisão — {NOME_CENARIO} (até 4 níveis; profundidade total {arvore.get_depth()})")
plt.show()

### Regras de decisão em texto (até 4 níveis)

In [ ]:
print(export_text(arvore, feature_names=COLUNAS, max_depth=4))

## Como avaliar este modelo

- **Overfitting:** compare `acurácia_treino` com `acurácia_teste` e a distância entre as curvas de treino e validação.
- **Estabilidade:** veja `f1_cv_desvio` e a dispersão dos folds. O teste tem cerca de 850 linhas, então 0,01 de acurácia corresponde a cerca de 8 pedidos.
- **Tipo de erro:** na matriz normalizada, a taxa de *Rejected → Approved* indica quantos pedidos que deveriam ser rejeitados o modelo aprovaria.
- **Limiar:** a curva de limiar mostra como trocar precisão por recall sem treinar outro modelo.

Neste cenário:

- Veja quais variáveis assumem o topo da árvore e a importância das variáveis sem o score.
- Um `roc_auc` próximo de 0,5 e curvas de probabilidade sobrepostas indicam que as variáveis restantes quase não têm informação sobre a aprovação.
- Modelos com pouca informação tendem a favorecer a classe majoritária: observe a diferença entre `recall_aprovado` e `recall_rejeitado`.